# Part 1: Data + Classification (Pneumonia CNN)

**Before running:** (1) add the dataset *RSNA Pneumonia Detection Challenge* to this notebook (right side → Add Input → search it; you must join the competition on Kaggle first), (2) turn on a GPU (Settings → Accelerator → GPU T4/P100). Then run the cells top to bottom.

In [ ]:
!pip install -q pydicom

## 1. Write the scripts to disk

In [ ]:
%%writefile model.py
"""
10-layer CNN that is compatible with Class Activation Maps (CAM).

Rules from the paper that this follows:
  - 10 conv layers, 3x3 kernels, zero padding ("same"), ReLU
  - ONE Global Average Pooling layer, then ONE fully connected layer -> 2 classes
  - Part 2 (CAM) uses  model.features(x)  and  model.fc.weight[pneumonia_class]

NOTE: this file is shared with Person B (localization). Do not change the
class/method names without telling them.
"""
import torch
import torch.nn as nn

# (out_channels) per conv layer; "M" = 2x2 max pool.  10 convs in total.
CFG = [64, 64, "M", 128, 128, "M", 256, 256, 256, "M", 256, 256, 256]


class CAMNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2, use_bn=True):
        super().__init__()
        layers, c = [], in_channels
        for v in CFG:
            if v == "M":
                layers.append(nn.MaxPool2d(2))
            else:
                layers.append(nn.Conv2d(c, v, kernel_size=3, padding=1))
                if use_bn:
                    layers.append(nn.BatchNorm2d(v))
                layers.append(nn.ReLU(inplace=True))
                c = v
        self.conv = nn.Sequential(*layers)
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(c, num_classes)

    def features(self, x):
        """Output of the last conv block: (B, 256, H/8, W/8). Used for CAM."""
        return self.conv(x)

    def forward(self, x):
        f = self.features(x)
        return self.fc(self.gap(f).flatten(1))


if __name__ == "__main__":
    m = CAMNet(in_channels=1)
    x = torch.randn(2, 1, 128, 128)
    print("logits:", m(x).shape, "| features:", m.features(x).shape)
    print("conv layers:", sum(isinstance(l, nn.Conv2d) for l in m.conv))


In [ ]:
%%writefile preprocess.py
"""
Step 1: Build the dataset.

- Reads the RSNA Pneumonia Detection labels + DICOM images
- Keeps only "Lung Opacity" (pneumonia = 1) and "Normal" (healthy = 0)
  (drops "No Lung Opacity / Not Normal", as in the paper)
- Resizes to 128x128, stratified 70/20/10 train/val/test split
- Saves everything to one compressed .npz (uint8 images + labels + mean/std)

Usage:
    python preprocess.py --data_dir /path/to/rsna --out data/rsna_128.npz

Expected layout of --data_dir (Kaggle RSNA, stage 2 names):
    stage_2_train_labels.csv
    stage_2_detailed_class_info.csv
    stage_2_train_images/<patientId>.dcm
If your folder uses the stage_1 names, pass --prefix stage_1.
"""
import argparse
import os

import numpy as np
import pandas as pd
import pydicom
from PIL import Image
from sklearn.model_selection import train_test_split
from tqdm import tqdm

SIZE = 128


def load_image(path, size=SIZE):
    dcm = pydicom.dcmread(path)
    img = dcm.pixel_array.astype(np.float32)
    # scale to 0-255 per image, then resize
    img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255.0
    img = Image.fromarray(img.astype(np.uint8)).resize((size, size), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_dir", required=True)
    ap.add_argument("--out", default="data/rsna_128.npz")
    ap.add_argument("--prefix", default="stage_2")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    labels = pd.read_csv(os.path.join(args.data_dir, f"{args.prefix}_train_labels.csv"))
    info = pd.read_csv(os.path.join(args.data_dir, f"{args.prefix}_detailed_class_info.csv"))

    # one row per patient (labels csv has one row per bounding box)
    info = info.drop_duplicates("patientId")
    info = info[info["class"].isin(["Lung Opacity", "Normal"])].copy()
    info["label"] = (info["class"] == "Lung Opacity").astype(int)
    print("Class counts:\n", info["label"].value_counts())

    ids = info["patientId"].tolist()
    y = info["label"].values.astype(np.int64)

    img_dir = os.path.join(args.data_dir, f"{args.prefix}_train_images")
    X = np.stack([load_image(os.path.join(img_dir, f"{p}.dcm")) for p in tqdm(ids)])
    ids = np.array(ids)

    # 70 / 20 / 10 stratified split
    idx = np.arange(len(y))
    idx_tr, idx_tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=args.seed)
    idx_val, idx_te = train_test_split(
        idx_tmp, test_size=1 / 3, stratify=y[idx_tmp], random_state=args.seed
    )  # 30% * 2/3 = 20% val, 30% * 1/3 = 10% test

    # per-pixel mean/std computed on TRAIN only (paper: normalise "at that location")
    mean = X[idx_tr].astype(np.float32).mean(axis=0)
    std = X[idx_tr].astype(np.float32).std(axis=0) + 1e-6

    os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
    np.savez_compressed(
        args.out,
        X_train=X[idx_tr], y_train=y[idx_tr], id_train=ids[idx_tr],
        X_val=X[idx_val], y_val=y[idx_val], id_val=ids[idx_val],
        X_test=X[idx_te], y_test=y[idx_te], id_test=ids[idx_te],
        mean=mean, std=std,
    )
    print(f"Saved {args.out}: train={len(idx_tr)} val={len(idx_val)} test={len(idx_te)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile baselines.py
"""
Step 2: Baselines - Logistic Regression, SVM, Random Forest on flattened pixels.

Usage:
    python baselines.py --data data/rsna_128.npz --size 64

--size downsamples the images before flattening so the SVM finishes in minutes.
Use --size 128 to match the paper exactly (16,384 features; SVM will be slow).

Output: baseline_results.json  (train/test accuracy for each classifier)
"""
import argparse
import json
import time

import numpy as np
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC


def prep(X, mean, std, size):
    X = (X.astype(np.float32) - mean) / std
    if size != X.shape[1]:
        X = np.stack([np.asarray(Image.fromarray(x).resize((size, size), Image.BILINEAR)) for x in X])
    return X.reshape(len(X), -1)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/rsna_128.npz")
    ap.add_argument("--size", type=int, default=64)
    ap.add_argument("--out", default="baseline_results.json")
    args = ap.parse_args()

    d = np.load(args.data)
    mean, std = d["mean"], d["std"]
    Xtr, ytr = prep(d["X_train"], mean, std, args.size), d["y_train"]
    Xte, yte = prep(d["X_test"], mean, std, args.size), d["y_test"]
    print("Feature vector length:", Xtr.shape[1])

    models = {
        "LR": LogisticRegression(max_iter=1000),
        "SVM": SVC(kernel="rbf"),
        "RF": RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42),
    }
    results = {}
    for name, clf in models.items():
        t = time.time()
        clf.fit(Xtr, ytr)
        results[name] = {
            "train_acc": float(clf.score(Xtr, ytr)),
            "test_acc": float(clf.score(Xte, yte)),
        }
        print(f"{name}: {results[name]}  ({time.time() - t:.0f}s)")

    json.dump(results, open(args.out, "w"), indent=2)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile train.py
"""
Step 3: Train the CNN and evaluate it.

Usage:
    python train.py --data data/rsna_128.npz --epochs 20 --lr 1e-4

Outputs (in --out_dir, default models/):
    cnn_best.pt          best-validation checkpoint (hand this to Person B)
    history.json         per-epoch train/val loss + accuracy
    training_curves.png  accuracy curve for the write-up
    test_report.json     test accuracy, confusion matrix, sensitivity, specificity
"""
import argparse
import json
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader, TensorDataset

from model import CAMNet


def make_loader(X, y, mean, std, batch_size, shuffle):
    X = (X.astype(np.float32) - mean) / std                # per-pixel normalisation
    X = torch.from_numpy(X).unsqueeze(1)                    # (N, 1, 128, 128)
    y = torch.from_numpy(y)
    return DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=shuffle)


@torch.no_grad()
def evaluate(model, loader, device, criterion=None):
    model.eval()
    preds, gts, total_loss = [], [], 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        if criterion is not None:
            total_loss += criterion(out, yb).item() * len(yb)
        preds.append(out.argmax(1).cpu().numpy())
        gts.append(yb.cpu().numpy())
    preds, gts = np.concatenate(preds), np.concatenate(gts)
    return (preds == gts).mean(), total_loss / len(gts), preds, gts


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", default="data/rsna_128.npz")
    ap.add_argument("--out_dir", default="models")
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--batch_size", type=int, default=64)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", device)

    d = np.load(args.data)
    mean, std = d["mean"], d["std"]
    train_dl = make_loader(d["X_train"], d["y_train"], mean, std, args.batch_size, True)
    val_dl = make_loader(d["X_val"], d["y_val"], mean, std, args.batch_size, False)
    test_dl = make_loader(d["X_test"], d["y_test"], mean, std, args.batch_size, False)

    model = CAMNet(in_channels=1).to(device)
    criterion = nn.CrossEntropyLoss()
    opt = torch.optim.Adam(model.parameters(), lr=args.lr)

    os.makedirs(args.out_dir, exist_ok=True)
    hist = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
    best_val = 0.0

    for ep in range(1, args.epochs + 1):
        model.train()
        correct, n, loss_sum = 0, 0, 0.0
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            opt.step()
            loss_sum += loss.item() * len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            n += len(yb)
        tr_acc, tr_loss = correct / n, loss_sum / n
        val_acc, val_loss, _, _ = evaluate(model, val_dl, device, criterion)

        for k, v in zip(hist, [tr_acc, val_acc, tr_loss, val_loss]):
            hist[k].append(float(v))
        print(f"Epoch {ep:02d}/{args.epochs} | train acc {tr_acc:.4f} loss {tr_loss:.4f} "
              f"| val acc {val_acc:.4f} loss {val_loss:.4f}")

        if val_acc > best_val:
            best_val = val_acc
            torch.save(
                {
                    "state_dict": model.state_dict(),
                    "in_channels": 1,
                    "mean": mean,
                    "std": std,
                    "val_acc": float(val_acc),
                },
                os.path.join(args.out_dir, "cnn_best.pt"),
            )

    json.dump(hist, open(os.path.join(args.out_dir, "history.json"), "w"), indent=2)

    # training curves (for the write-up / slides)
    plt.figure(figsize=(6, 4))
    plt.plot(hist["train_acc"], label="train")
    plt.plot(hist["val_acc"], label="validation")
    plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.title("Training and validation accuracy")
    plt.legend(); plt.tight_layout()
    plt.savefig(os.path.join(args.out_dir, "training_curves.png"), dpi=150)

    # final evaluation with the best checkpoint
    ckpt = torch.load(os.path.join(args.out_dir, "cnn_best.pt"), weights_only=False)
    model.load_state_dict(ckpt["state_dict"])
    test_acc, _, preds, gts = evaluate(model, test_dl, device)
    tn, fp, fn, tp = confusion_matrix(gts, preds, labels=[0, 1]).ravel()
    report = {
        "test_accuracy": float(test_acc),
        "confusion_matrix": {"TP": int(tp), "FN": int(fn), "FP": int(fp), "TN": int(tn)},
        "sensitivity_recall": float(tp / (tp + fn)),
        "specificity": float(tn / (tn + fp)),
        "precision": float(tp / (tp + fp)) if (tp + fp) else 0.0,
    }
    json.dump(report, open(os.path.join(args.out_dir, "test_report.json"), "w"), indent=2)
    print("\nTEST:", json.dumps(report, indent=2))


if __name__ == "__main__":
    main()


## 2. Find the dataset folder

In [ ]:
import glob, os
hits = glob.glob('/kaggle/input/**/*_train_labels.csv', recursive=True)
print(hits)
assert hits, 'Dataset not found: add the RSNA Pneumonia Detection Challenge data via Add Input'
DATA_DIR = os.path.dirname(hits[0])
PREFIX = os.path.basename(hits[0]).replace('_train_labels.csv', '')
print('DATA_DIR =', DATA_DIR, '| PREFIX =', PREFIX)

## 3. Preprocess (reads DICOMs, resizes to 128x128, 70/20/10 split) - takes several minutes

In [ ]:
!python preprocess.py --data_dir {DATA_DIR} --prefix {PREFIX} --out data/rsna_128.npz

## 4. Baselines (LR, SVM, RF)

In [ ]:
!python baselines.py --data data/rsna_128.npz --size 64

## 5. Train the CNN (20 epochs, Adam, lr 1e-4)

In [ ]:
!python train.py --data data/rsna_128.npz --epochs 20 --lr 1e-4

## 6. Results to copy into the write-up

In [ ]:
import json
print(json.load(open('baseline_results.json')))
print(json.load(open('models/test_report.json')))
from IPython.display import Image
Image('models/training_curves.png')

Download from the **Output** tab / file browser: `models/cnn_best.pt`, `models/training_curves.png`, `models/test_report.json`, `baseline_results.json`, `model.py`.